# Trisha-v.4.1: 1-Stage Full 3 Classes + Compound Scaling (Fixed)
5 model (EfficientFormer-L1, ConvNeXt-Tiny, CLIP ViT-B/32, DINOv2 ViT-B/14, ConvNeXt-Small).
- **1-Stage Full 3 Classes** (no Stage 2) — hindari catastrophic forgetting
- **192px → 256px** compound scaling — training stabil, inference resolusi tinggi
- **ClassBalancedLoss** + **RandAugment** — atasi imbalance + augmentasi kuat
- **TTA** (5-crop + flip) + **Weighted Ensemble** — final inference
- **Data:** train/ → train_clean_v4/ (rename + dedup only, no test leakage)
- **Fixes:** Swin-Tiny → ConvNeXt-Small (window alignment), Phase 3 loss → ClassBalancedLoss + LabelSmoothing, EMA, +DINOv2 ViT-B/14
- **Output:** `experiments/results/trisha_v4.0/`

In [1]:
import hashlib
import json
import os
import shutil
import sys
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

%matplotlib inline

_cwd = Path.cwd()
if (_cwd / 'modules').exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd.parent / 'modules').exists():
    sys.path.insert(0, str(_cwd.parent))
else:
    sys.path.insert(0, str(_cwd.parent.parent))

: 

In [2]:
try:
    import open_clip
    print('open-clip-torch already installed')
except ImportError:
    print('open-clip-torch not available - CLIP model will be skipped')
    open_clip = None

d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


open-clip-torch already installed


In [4]:
from modules.utils.config import CLASS_LABELS, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.models.factory import TrashClassifier
from modules.models.clip_adapter import CLIPAdapter
from modules.training.loss import ClassBalancedLoss
from modules.training.train import fit
from modules.training.collator import MixUpCollator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = RESULTS / 'trisha_v4.0'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')
print(f'Project root: {PROJECT_ROOT}')

Device: cuda
Output dir: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v4.0
Project root: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble


---
## 1. Setup Data: train_clean/
Copy `train/` → `train_clean_v4/`, rename 3 long-path, MD5 dedup dalam train_clean/.

In [5]:
SRC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean_v4'

if not SRC_DIR.exists():
    raise FileNotFoundError('train/ not found. Download from drive first.')

if TRAIN_DIR.exists():
    print(f'{TRAIN_DIR} already exists, skipping copy.')
else:
    print('Copying train/ -> train_clean_v4/...')
    ret = os.system(f'robocopy "{SRC_DIR}" "{TRAIN_DIR}" /E /COPY:DAT /R:2 /W:2 /NDL /NFL /NJH /NJS')
    print('Done')

Copying train/ -> train_clean_v4/...
Done


In [6]:
def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

long_names = [
    ("1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-representasi-visual-yang-luar-biasa-dari-masalah-limbah-elektronik-yang-berkembang-dan-kebutuhan-akan-solusi-daur-ulang-berkelanjutan_73523-11403.jpg", "E_001.jpg"),
    ("12.pile-discarded-motherboards-cpus-cables-disc-drives-hijacked-hardware-heap-concept-electronic-waste-tech-recycling-hardware-reuse-sustainable-technology-environmental-impact_864588-263287.jpg", "E_012.jpg"),
    ("64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat-seperti-perangkat-elektronik-yang-mengeluarkan-bahan-kimia-beracun-ke-dalam-tanah-dan-saluran-air-di-lokasi-pembuangan-ilegal_997534-43151.jpg", "E_064.jpg"),
]
for old_name, new_name in long_names:
    old = TRAIN_DIR / '1_Electronic' / old_name
    new = old.with_name(new_name)
    try:
        os.rename(_safe_path(old), _safe_path(new))
        print(f'{old_name[:40]}... -> {new_name}')
    except FileNotFoundError:
        pass
    except FileExistsError:
        pass
print('Rename done.')

# MD5 dedup within train_clean/
hash_map = defaultdict(list)
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        try:
            with open(_safe_path(fpath), 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})
        except:
            pass

dup_groups = {h: files for h, files in hash_map.items() if len(files) > 1}
print(f'Duplicate groups: {len(dup_groups)}')

removed = 0
for h, files in dup_groups.items():
    for f in files[1:]:
        os.remove(f['path'])
        removed += 1
print(f'Removed {removed} duplicate files.')

1.tumpukan-e-limbah-yang-kacau-dari-lapt... -> E_001.jpg
12.pile-discarded-motherboards-cpus-cabl... -> E_012.jpg
64.dampak-lingkungan-dari-ewaste-tangkap... -> E_064.jpg
Rename done.
Duplicate groups: 62
Removed 62 duplicate files.


---
## 2. Load & Split Data (train_clean/)

In [7]:
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean_v4'
records = []
for label_dir in sorted(TRAIN_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    for img in sorted(label_dir.glob('*')):
        if img.is_file():
            records.append({'path': str(img.relative_to(PROJECT_ROOT)), 'label': label_dir.name})
df = pd.DataFrame(records)
print(f'Total: {len(df)}')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

Total: 26465
  0_Recyclable: 9990
  1_Electronic: 3931
  2_Organic: 12544
Train: 21172, Val: 5293


---
## 3. Dataset + Transforms (192px & 256px)
RandAugment + compound scaling: training 192px, fine-tune 256px.

In [8]:
_LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}

def _open_image(path):
    safe = _safe_path(path)
    with open(safe, 'rb') as f:
        return Image.open(BytesIO(f.read())).convert('RGB')

class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = (PROJECT_ROOT / df['path']).values
        self.labels = df['label'].map(_LABEL_TO_IDX).values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = _open_image(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def make_transform(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.3, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.25),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def make_loader(df, img_size, batch_size=32, shuffle=True, use_mixup=False, num_workers=0):
    ds = TrashDataset(df, transform=make_transform(img_size, is_train=shuffle))
    collator = MixUpCollator(alpha=0.2, num_classes=3) if (use_mixup and shuffle) else None
    sampler = None
    if shuffle:
        labels = df['label'].map(_LABEL_TO_IDX).values
        counts = torch.bincount(torch.tensor(labels))
        weights = 1.0 / counts[labels].float()
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        ds, batch_size=batch_size,
        sampler=sampler, shuffle=False if sampler else shuffle,
        num_workers=num_workers, pin_memory=False,
        collate_fn=collator,
    )

---
## 4. Dataloaders
- Phase 1 & 2: 192px (batch 32)
- Phase 3: 256px (batch 16, CLIP batch 8)

In [9]:
# Phase 1 & 2: 192px
train_loader_192 = make_loader(train_df, img_size=192, batch_size=32, use_mixup=True)
val_loader_192 = make_loader(val_df, img_size=192, batch_size=32, shuffle=False)
print(f'192px - Train: {len(train_loader_192.dataset)}, Val: {len(val_loader_192.dataset)}')

# CLIP needs smaller batch for 192px too
train_loader_192_clip = make_loader(train_df, img_size=192, batch_size=16, use_mixup=True)
val_loader_192_clip = make_loader(val_df, img_size=192, batch_size=16, shuffle=False)

# Phase 3: 256px (compound scaling)
train_loader_256 = make_loader(train_df, img_size=256, batch_size=16, use_mixup=False)
val_loader_256 = make_loader(val_df, img_size=256, batch_size=16, shuffle=False)
print(f'256px - Train: {len(train_loader_256.dataset)}, Val: {len(val_loader_256.dataset)}')

# CLIP needs smaller batch for 256px
train_loader_256_clip = make_loader(train_df, img_size=256, batch_size=8, use_mixup=False)
val_loader_256_clip = make_loader(val_df, img_size=256, batch_size=8, shuffle=False)

C:\Users\Acer\AppData\Local\Temp\ipykernel_13744\2957976876.py:45: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  weights = 1.0 / counts[labels].float()


192px - Train: 21172, Val: 5293
256px - Train: 21172, Val: 5293


---
## 5. Helper Functions
eval_model, TTA, move_to_outdir, ensemble.

In [10]:
def move_to_outdir(name):
    for ext in ['.pt', '.json']:
        src = RESULTS / f'{name}{ext}'
        if src.exists():
            shutil.move(str(src), str(OUT_DIR / f'{name}{ext}'))
    src_ema = RESULTS / f'{name}_ema.pt'
    if src_ema.exists():
        shutil.move(str(src_ema), str(OUT_DIR / f'{name}_ema.pt'))

def eval_model(model, loader):
    model.to(DEVICE).eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='Eval', leave=False):
            out = model(x.to(DEVICE))
            preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(labels, preds, average='macro')
    f1_pc = f1_score(labels, preds, average=None, labels=list(range(3))).tolist()
    prec_pc = precision_score(labels, preds, average=None, labels=list(range(3))).tolist()
    rec_pc = recall_score(labels, preds, average=None, labels=list(range(3))).tolist()
    return {'f1_macro': f1, 'f1_per_class': f1_pc, 'precision': prec_pc, 'recall': rec_pc, 'preds': preds, 'labels': labels}

def tta_predict(model, val_df, img_size=256):
    """Test-time augmentation: 5-crop + horizontal flip -> average softmax."""
    model.to(DEVICE).eval()

    ds = TrashDataset(val_df, transform=None)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)

    tta_transform = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.TenCrop(img_size),
        transforms.Lambda(lambda crops: torch.stack([
            transforms.Normalize(MEAN, STD)(transforms.ToTensor()(crop))
            for crop in crops
        ])),
    ])
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='TTA', leave=False):
            crops = tta_transform(x[0]).to(DEVICE)  # [10, 3, H, W]
            logits = model(crops)  # [10, 3]
            probs = F.softmax(logits, dim=1).mean(dim=0)  # average over 10 crops
            all_probs.append(probs.cpu())
            all_labels.append(y.item())
    all_probs = torch.stack(all_probs)
    preds = all_probs.argmax(dim=1).numpy().tolist()
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(all_labels, preds, average='macro')
    f1_pc = f1_score(all_labels, preds, average=None, labels=list(range(3))).tolist()
    return {
        'f1_macro': f1, 'f1_per_class': f1_pc, 'preds': preds, 'labels': all_labels,
        'probs': all_probs,
    }

def weighted_ensemble(results_list, weights=None):
    """Weighted ensemble by averaging softmax probs."""
    if weights is None:
        weights = [1.0 / len(results_list)] * len(results_list)
    weights = torch.tensor(weights) / sum(weights)
    probs = sum(w * r['probs'] for w, r in zip(weights, results_list))
    preds = probs.argmax(dim=1).numpy().tolist()
    labels = results_list[0]['labels']
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(labels, preds, average='macro')
    f1_pc = f1_score(labels, preds, average=None, labels=list(range(3))).tolist()
    prec_pc = precision_score(labels, preds, average=None, labels=list(range(3))).tolist()
    rec_pc = recall_score(labels, preds, average=None, labels=list(range(3))).tolist()
    return {'f1_macro': f1, 'f1_per_class': f1_pc, 'precision': prec_pc, 'recall': rec_pc, 'preds': preds, 'labels': labels}

---
## 6. TRAIN: EfficientFormer-L1
P1: 192px head-only (10 ep) → P2: 192px fine-tune (25 ep) → P3: 256px fine-tune (10 ep)

In [11]:
MODEL = 'efficientformer_l1'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))

# Phase 1: 192px, Head Only, ClassBalancedLoss
print(f'\nPhase 1: 192px, Head Only @ lr=5e-4')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=10, epochs_finetune=0,
    lr_head=5e-4, lr_finetune=0, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p1')

# Phase 2: 192px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p1.pt', map_location='cpu'))
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
print(f'\nPhase 2: 192px, Fine-tune All @ lr=5e-5')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=25,
    lr_head=0, lr_finetune=5e-5, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p2')

# Phase 3: 256px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p2.pt', map_location='cpu'))
print(f'\nPhase 3: 256px, Fine-tune All @ lr=1e-5')
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
fit(model, train_loader_256, val_loader_256,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=10,
    lr_head=0, lr_finetune=1e-5, patience=4,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval: standard + TTA
print(f'\nFinal eval:')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_256)
print(f'Standard val: F1={res["f1_macro"]:.4f}')
res_tta = tta_predict(model, val_df, img_size=256)
print(f'TTA val: F1={res_tta["f1_macro"]:.4f}')
print(classification_report(res_tta['labels'], res_tta['preds'], target_names=CLASS_LABELS, digits=4))

# Save final
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
with open(OUT_DIR / f'{MODEL}_tta.json', 'w') as f:
    json.dump({k: v for k, v in res_tta.items() if k not in ['preds', 'labels', 'probs']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== efficientformer_l1 ===
Parameters: 11,623,355

Phase 1: 192px, Head Only @ lr=5e-4

=== efficientformer_l1_p1: Phase 1 — Head Only ===


efficientformer_l1_p1 train:   0%|          | 0/662 [00:00<?, ?it/s]

: 

---
## 7. TRAIN: ConvNeXt-Tiny
P1: 192px head-only (10 ep) → P2: 192px fine-tune (25 ep) → P3: 256px fine-tune (10 ep)

In [ ]:
MODEL = 'convnext_tiny'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))

# Phase 1: 192px, Head Only, ClassBalancedLoss
print(f'\nPhase 1: 192px, Head Only @ lr=5e-4')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=10, epochs_finetune=0,
    lr_head=5e-4, lr_finetune=0, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p1')

# Phase 2: 192px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p1.pt', map_location='cpu'))
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
print(f'\nPhase 2: 192px, Fine-tune All @ lr=5e-5')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=25,
    lr_head=0, lr_finetune=5e-5, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p2')

# Phase 3: 256px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p2.pt', map_location='cpu'))
print(f'\nPhase 3: 256px, Fine-tune All @ lr=1e-5')
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
fit(model, train_loader_256, val_loader_256,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=10,
    lr_head=0, lr_finetune=1e-5, patience=4,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval: standard + TTA
print(f'\nFinal eval:')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_256)
print(f'Standard val: F1={res["f1_macro"]:.4f}')
res_tta = tta_predict(model, val_df, img_size=256)
print(f'TTA val: F1={res_tta["f1_macro"]:.4f}')
print(classification_report(res_tta['labels'], res_tta['preds'], target_names=CLASS_LABELS, digits=4))

# Save final
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
with open(OUT_DIR / f'{MODEL}_tta.json', 'w') as f:
    json.dump({k: v for k, v in res_tta.items() if k not in ['preds', 'labels', 'probs']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 8. TRAIN: CLIP ViT-B/32 (VLM)
P1: 192px head-only (10 ep) → P2: 192px fine-tune (15 ep) → P3: 256px fine-tune (8 ep)

In [ ]:
class CLIPTrainable(CLIPAdapter):
    @property
    def classifier(self):
        return self.visual_projection
    @property
    def encoder(self):
        return self.clip.visual
    def freeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = True

MODEL = 'clip_vit_b32'
print(f'=== {MODEL} ===')

model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
clip_params = sum(p.numel() for p in model.clip.visual.parameters() if p.requires_grad)
proj_params = sum(p.numel() for p in model.visual_projection.parameters())
print(f'CLIP encoder: {clip_params:,}, Projection: {proj_params:,}')

# Zero-shot benchmark
model.to(DEVICE).eval()
prompts = ['a photo of recyclable waste', 'a photo of electronic waste', 'a photo of organic waste']
zs_preds, zs_labels = [], []
for x, y in tqdm(val_loader_192, desc='Zero-shot', leave=False):
    logits = model.zero_shot_classify(x.to(DEVICE), prompts)
    zs_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    zs_labels.extend(y.numpy().tolist())
from sklearn.metrics import f1_score
print(f'Zero-shot F1 macro: {f1_score(zs_labels, zs_preds, average="macro"):.4f}')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))

# Phase 1: 192px, Head Only, ClassBalancedLoss
print(f'\nPhase 1: 192px, Head Only @ lr=1e-3')
fit(model, train_loader_192_clip, val_loader_192_clip,
    name=f'{MODEL}_p1', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=10, epochs_finetune=0,
    lr_head=1e-3, lr_finetune=0, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p1')

# Phase 2: 192px, Fine-tune All, ClassBalancedLoss
model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p1.pt', map_location='cpu'))
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
print(f'\nPhase 2: 192px, Fine-tune All @ lr=1e-5')
fit(model, train_loader_192_clip, val_loader_192_clip,
    name=f'{MODEL}_p2', encoder_name=MODEL,
    accumulation_steps=8, epochs_head=0, epochs_finetune=15,
    lr_head=0, lr_finetune=1e-5, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p2')

# Phase 3: 256px, Fine-tune All, ClassBalancedLoss
model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p2.pt', map_location='cpu'))
print(f'\nPhase 3: 256px, Fine-tune All @ lr=1e-6')
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
fit(model, train_loader_256_clip, val_loader_256_clip,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=1e-6, patience=4,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval: standard + TTA
print(f'\nFinal eval:')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_256_clip)
print(f'Standard val: F1={res["f1_macro"]:.4f}')
res_tta = tta_predict(model, val_df, img_size=256)
print(f'TTA val: F1={res_tta["f1_macro"]:.4f}')
print(classification_report(res_tta['labels'], res_tta['preds'], target_names=CLASS_LABELS, digits=4))

# Save final
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
with open(OUT_DIR / f'{MODEL}_tta.json', 'w') as f:
    json.dump({k: v for k, v in res_tta.items() if k not in ['preds', 'labels', 'probs']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 9. TRAIN: DINOv2 ViT-B/14
P1: 192px head-only (10 ep) → P2: 192px fine-tune (20 ep) → P3: 256px fine-tune (8 ep)

In [ ]:
MODEL = 'vit_base_patch14_reg4_dinov2.lvd142m'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))

# Phase 1: 192px, Head Only, ClassBalancedLoss
print(f'\nPhase 1: 192px, Head Only @ lr=5e-4')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=10, epochs_finetune=0,
    lr_head=5e-4, lr_finetune=0, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p1')

# Phase 2: 192px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p1.pt', map_location='cpu'))
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
print(f'\nPhase 2: 192px, Fine-tune All @ lr=5e-5')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=20,
    lr_head=0, lr_finetune=5e-5, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p2')

# Phase 3: 256px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p2.pt', map_location='cpu'))
print(f'\nPhase 3: 256px, Fine-tune All @ lr=1e-5')
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
fit(model, train_loader_256, val_loader_256,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=1e-5, patience=4,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval: standard + TTA
print(f'\nFinal eval:')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_256)
print(f'Standard val: F1={res["f1_macro"]:.4f}')
res_tta = tta_predict(model, val_df, img_size=256)
print(f'TTA val: F1={res_tta["f1_macro"]:.4f}')
print(classification_report(res_tta['labels'], res_tta['preds'], target_names=CLASS_LABELS, digits=4))

# Save final
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
with open(OUT_DIR / f'{MODEL}_tta.json', 'w') as f:
    json.dump({k: v for k, v in res_tta.items() if k not in ['preds', 'labels', 'probs']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 10. TRAIN: ConvNeXt-Small
P1: 192px head-only (10 ep) → P2: 192px fine-tune (25 ep) → P3: 256px fine-tune (10 ep)

In [ ]:
MODEL = 'convnext_small'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))

# Phase 1: 192px, Head Only, ClassBalancedLoss
print(f'\nPhase 1: 192px, Head Only @ lr=5e-4')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=10, epochs_finetune=0,
    lr_head=5e-4, lr_finetune=0, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p1')

# Phase 2: 192px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p1.pt', map_location='cpu'))
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
print(f'\nPhase 2: 192px, Fine-tune All @ lr=5e-5')
fit(model, train_loader_192, val_loader_192,
    name=f'{MODEL}_p2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=25,
    lr_head=0, lr_finetune=5e-5, patience=6,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_p2')

# Phase 3: 256px, Fine-tune All, ClassBalancedLoss
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_p2.pt', map_location='cpu'))
print(f'\nPhase 3: 256px, Fine-tune All @ lr=1e-5')
criterion.update(torch.tensor(train_df['label'].map(_LABEL_TO_IDX).values))
fit(model, train_loader_256, val_loader_256,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=0, epochs_finetune=10,
    lr_head=0, lr_finetune=1e-5, patience=4,
    criterion=criterion, device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval: standard + TTA
print(f'\nFinal eval:')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_256)
print(f'Standard val: F1={res["f1_macro"]:.4f}')
res_tta = tta_predict(model, val_df, img_size=256)
print(f'TTA val: F1={res_tta["f1_macro"]:.4f}')
print(classification_report(res_tta['labels'], res_tta['preds'], target_names=CLASS_LABELS, digits=4))

# Save final
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
with open(OUT_DIR / f'{MODEL}_tta.json', 'w') as f:
    json.dump({k: v for k, v in res_tta.items() if k not in ['preds', 'labels', 'probs']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 11. Weighted Ensemble + Summary Table
Weighted ensemble based on TTA val F1.

In [ ]:
model_configs = [
    {'name': 'efficientformer_l1', 'batch': 16},
    {'name': 'convnext_tiny', 'batch': 16},
    {'name': 'clip_vit_b32', 'batch': 8},
    {'name': 'vit_base_patch14_reg4_dinov2.lvd142m', 'batch': 16},
    {'name': 'convnext_small', 'batch': 16},
]

rows = []
tta_results = {}

for cfg in model_configs:
    m = cfg['name']
    print(f'\n=== {m} ===')

    # Load model
    if m == 'clip_vit_b32':
        model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
    else:
        model = TrashClassifier(encoder_name=m, num_classes=3)

    path_final = OUT_DIR / f'{m}.pt'
    if path_final.exists():
        model.load_state_dict(torch.load(path_final, map_location=DEVICE))
        model.to(DEVICE).eval()
    else:
        print(f'  {m}.pt not found, skipping')
        continue

    # TTA eval on val
    res = tta_predict(model, val_df, img_size=256)
    tta_results[m] = res

    print(f'  F1 macro: {res["f1_macro"]:.4f}')
    print(f'  F1: R={res["f1_per_class"][0]:.4f} E={res["f1_per_class"][1]:.4f} O={res["f1_per_class"][2]:.4f}')

    rows.append({
        'model': m,
        'params': sum(p.numel() for p in model.parameters()),
        'f1_macro': res['f1_macro'],
        'f1_0_recyclable': res['f1_per_class'][0],
        'f1_1_electronic': res['f1_per_class'][1],
        'f1_2_organic': res['f1_per_class'][2],
    })

# Weighted ensemble
if len(tta_results) >= 2:
    print(f'\n=== Weighted Ensemble ===')
    weights = [r['f1_macro'] ** 2 for r in tta_results.values()]
    print(f'Weights: {[f"{w:.4f}" for w in weights]}')
    ensemble_res = weighted_ensemble(list(tta_results.values()), weights=weights)
    print(f'\nEnsemble F1 macro: {ensemble_res["f1_macro"]:.4f}')
    print(classification_report(ensemble_res['labels'], ensemble_res['preds'],
                                target_names=CLASS_LABELS, digits=4))
    ensemble_res['weights'] = [float(w) for w in weights]
    with open(OUT_DIR / 'ensemble_val.json', 'w') as f:
        json.dump({k: v for k, v in ensemble_res.items() if k not in ['preds', 'labels']}, f, indent=2)

# Summary table
df_result = pd.DataFrame(rows).sort_values('f1_macro', ascending=False)
print(f'\n=== Summary ===')
print(df_result.to_string(index=False))
df_result.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f'\nSaved: {OUT_DIR / "summary.csv"}')